In [ ]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import roc_auc_score
from tqdm import tqdm


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [ ]:
CFG = {
    "img_size": 224,              # change to 456 if EfficientNet-B5
    "batch_size": 8,
    "epochs": 100,
    "lr": 1e-4,
    "num_workers": 2,
    "patience": 20,
    "label_smoothing": 0.1
}


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
class DDIFusionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        # Image
        img_path = os.path.join(self.img_dir, row["image_name"])
        image = torchvision.io.read_image(img_path).float() / 255.0
        image = transforms.ToPILImage()(image)

        if self.transform:
            image = self.transform(image)

        # ---- METADATA ----
        # skin_tone: {0,1,2}
        skin_tone = torch.zeros(3)
        skin_tone[row["skin_tone"]] = 1.0

        # cancer_type: {0,1}
        cancer_type = torch.zeros(2)
        cancer_type[row["cancer_type"]] = 1.0

        metadata = torch.cat([skin_tone, cancer_type], dim=0)

        label = torch.tensor(row["malignant"], dtype=torch.float32)

        return image, metadata, label


In [ ]:
train_dataset = DDIFusionDataset(
    train_df,
    img_dir=TRAIN_IMG_DIR,
    transform=train_transform
)

val_dataset = DDIFusionDataset(
    val_df,
    img_dir=TRAIN_IMG_DIR,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"]
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"]
)


NameError: name 'train_df' is not defined

In [ ]:
class FeatureFusionModel(nn.Module):
    def __init__(self, backbone="densenet121", meta_dim=5):
        super().__init__()

        if backbone == "densenet121":
            base_model = models.densenet121(pretrained=True)
            self.feature_extractor = base_model.features
            img_feat_dim = 1024

        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d(1)

        # ---- CLASSIFIER ----
        self.classifier = nn.Sequential(
            nn.Linear(img_feat_dim + meta_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(512, 1)
        )

    def forward(self, image, metadata):
        x = self.feature_extractor(image)
        x = self.pool(x)
        x = x.view(x.size(0), -1)

        fused = torch.cat([x, metadata], dim=1)
        out = self.classifier(fused)

        return out


In [ ]:
model = FeatureFusionModel(backbone="densenet121").to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=CFG["lr"])


In [ ]:
def smooth_labels(labels, smoothing=0.1):
    return labels * (1 - smoothing) + 0.5 * smoothing


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, metadata, labels in tqdm(loader, desc="Training", leave=False):
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.to(device).unsqueeze(1)

        labels = smooth_labels(labels, CFG["label_smoothing"])

        optimizer.zero_grad()

        logits = model(images, metadata)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_targets.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_auc = roc_auc_score(all_targets, all_preds)

    return epoch_loss, epoch_auc


In [ ]:
@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, metadata, labels in tqdm(loader, desc="Validation", leave=False):
        images = images.to(device)
        metadata = metadata.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(images, metadata)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_targets.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_auc = roc_auc_score(all_targets, all_preds)

    return epoch_loss, epoch_auc


In [ ]:
class EarlyStopping:
    def __init__(self, patience=20, mode="max"):
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return True

        improvement = score > self.best_score if self.mode == "max" else score < self.best_score

        if improvement:
            self.best_score = score
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False


In [ ]:
best_auc = 0.0
early_stopper = EarlyStopping(patience=CFG["patience"], mode="max")

history = {
    "train_loss": [],
    "train_auc": [],
    "val_loss": [],
    "val_auc": []
}

for epoch in range(CFG["epochs"]):
    print(f"\nEpoch [{epoch+1}/{CFG['epochs']}]")

    train_loss, train_auc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_auc = validate_one_epoch(
        model,
        val_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_auc"].append(train_auc)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)

    print(
        f"Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f} || "
        f"Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f}"
    )

    improved = early_stopper.step(val_auc)

    if improved:
        best_auc = val_auc
        torch.save(model.state_dict(), "best_feature_fusion_model.pth")
        print("✅ Best model saved")

    if early_stopper.early_stop:
        print("⏹ Early stopping triggered")
        break


In [ ]:
model.load_state_dict(torch.load("best_feature_fusion_model.pth"))
print(f"Best Validation AUC: {best_auc:.4f}")


In [ ]:
# =========================
# FEATURE FUSION – ALL-IN-ONE CELL
# =========================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Reproducibility
# -------------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Backbone selector (same style as your notebook)
# -------------------------
def get_backbone(backbone_name):
    if backbone_name == "densenet121":
        model = torchvision.models.densenet121(pretrained=True)
        feature_extractor = model.features
        feat_dim = 1024
        img_size = 224

    elif backbone_name == "efficientnet_b5":
        model = torchvision.models.efficientnet_b5(pretrained=True)
        feature_extractor = model.features
        feat_dim = 2048
        img_size = 456

    elif backbone_name == "inception_v3":
        model = torchvision.models.inception_v3(
            pretrained=True,
            aux_logits=False
        )
        feature_extractor = nn.Sequential(*list(model.children())[:-1])
        feat_dim = 2048
        img_size = 299

    else:
        raise ValueError(f"Unsupported backbone: {backbone_name}")

    return feature_extractor, feat_dim, img_size

# -------------------------
# CONFIG (change backbone here)
# -------------------------
BACKBONE = "densenet121"   # "efficientnet_b5" | "inception_v3"

feature_extractor, IMG_FEAT_DIM, IMG_SIZE = get_backbone(BACKBONE)

CFG = {
    "img_size": IMG_SIZE,
    "batch_size": 8,
    "lr": 1e-4
}

# -------------------------
# Transforms
# -------------------------
train_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -------------------------
# Dataset (IMAGE + METADATA)
# -------------------------
class DDIFusionDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img_path = os.path.join(self.img_dir, row["image_name"])
        image = torchvision.io.read_image(img_path).float() / 255.0
        image = transforms.ToPILImage()(image)

        if self.transform:
            image = self.transform(image)

        # Metadata: skin tone (3) + cancer type (2)
        skin_tone = torch.zeros(3)
        skin_tone[row["skin_tone"]] = 1.0

        cancer_type = torch.zeros(2)
        cancer_type[row["cancer_type"]] = 1.0

        metadata = torch.cat([skin_tone, cancer_type], dim=0)

        label = torch.tensor(row["malignant"], dtype=torch.float32)

        return image, metadata, label

# -------------------------
# Feature Fusion Model (Figure 1B)
# -------------------------
class FeatureFusionModel(nn.Module):
    def __init__(self, backbone_name, meta_dim=5):
        super().__init__()

        self.feature_extractor, img_feat_dim, _ = get_backbone(backbone_name)

        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(img_feat_dim + meta_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(512, 1)
        )

    def forward(self, image, metadata):
        x = self.feature_extractor(image)
        x = self.pool(x)
        x = x.view(x.size(0), -1)

        fused = torch.cat([x, metadata], dim=1)
        return self.classifier(fused)

# -------------------------
# Model init (ONE-LINE SWITCH)
# -------------------------
model = FeatureFusionModel(BACKBONE).to(device)

print(f"✅ Feature Fusion Model Ready | Backbone: {BACKBONE} | Image size: {IMG_SIZE}")


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 205MB/s]


✅ Feature Fusion Model Ready | Backbone: densenet121 | Image size: 224


In [ ]:
# =========================
# FEATURE FUSION – TESTING (SINGLE CELL)
# =========================

import os
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Load trained model
# -------------------------
model = FeatureFusionModel(BACKBONE).to(device)
model.load_state_dict(torch.load("best_feature_fusion_model.pth", map_location=device))
model.eval()

# -------------------------
# Test-Time Augmentation (TTA)
# -------------------------
TTA_TRANSFORMS = [
    transforms.Compose([
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    transforms.Compose([
        transforms.Resize((CFG["img_size"], CFG["img_size"])),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
]

# -------------------------
# Test Dataset
# -------------------------
class DDIFusionTestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]

        img_path = os.path.join(self.img_dir, row["image_name"])
        image = torchvision.io.read_image(img_path).float() / 255.0
        image = transforms.ToPILImage()(image)
        image = self.transform(image)

        # Metadata
        skin_tone = torch.zeros(3)
        skin_tone[row["skin_tone"]] = 1.0

        cancer_type = torch.zeros(2)
        cancer_type[row["cancer_type"]] = 1.0

        metadata = torch.cat([skin_tone, cancer_type], dim=0)
        label = torch.tensor(row["malignant"], dtype=torch.float32)

        return image, metadata, label

# -------------------------
# TTA Inference
# -------------------------
@torch.no_grad()
def run_tta(model, df, img_dir, tta_transforms):
    all_probs = []
    all_labels = []

    for transform in tta_transforms:
        dataset = DDIFusionTestDataset(df, img_dir, transform)
        loader = DataLoader(dataset, batch_size=1, shuffle=False)

        probs = []

        for images, metadata, labels in tqdm(loader, leave=False):
            images = images.to(device)
            metadata = metadata.to(device)

            logits = model(images, metadata)
            prob = torch.sigmoid(logits).item()

            probs.append(prob)

        all_probs.append(probs)

    all_probs = np.mean(np.array(all_probs), axis=0)
    all_labels = df["malignant"].values

    return all_probs, all_labels

# -------------------------
# Run Testing
# -------------------------
test_probs, test_labels = run_tta(
    model,
    test_df,
    TEST_IMG_DIR,
    TTA_TRANSFORMS
)

test_auc = roc_auc_score(test_labels, test_probs)
print(f"✅ Test AUC (TTA): {test_auc:.4f}")
